# GitHub User READMEs Fetcher

이 노트북은 특정 GitHub ID를 입력받아 해당 사용자의 모든 공개 저장소(Public Repo) 목록을 가져온 뒤, 각 저장소의 `README.md` 내용을 수집하고 **로컬 파일로 저장**합니다.

In [ ]:
import httpx
import os
import base64
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  # .env 파일에서 GITHUB_TOKEN 등을 로드할 수 있습니다.

GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")  # API 제한을 피하려면 토큰을 사용하는 것이 좋습니다.
HEADERS = {"Accept": "application/vnd.github.v3+json"}
if GITHUB_TOKEN:
    HEADERS["Authorization"] = f"token {GITHUB_TOKEN}"

client = httpx.Client(headers=HEADERS, timeout=10.0)

In [ ]:
def get_user_repos(github_id):
    """사용자의 모든 공개 저장소 목록을 가져옵니다."""
    url = f"https://api.github.com/users/{github_id}/repos"
    params = {"type": "public", "per_page": 100}
    response = client.get(url, params=params)
    response.raise_for_status()
    return response.json()

def get_readme_content(full_name):
    """특정 저장소(user/repo)의 README 내용을 가져옵니다."""
    url = f"https://api.github.com/repos/{full_name}/readme"
    try:
        response = client.get(url)
        if response.status_code == 200:
            data = response.json()
            content = base64.b64decode(data['content']).decode('utf-8')
            return content
        else:
            return None
    except Exception as e:
        print(f"Error fetching README for {full_name}: {e}")
        return None

def save_readme(github_id, repo_name, content):
    """README 내용을 파일로 저장합니다."""
    output_dir = Path(f"extracted_readmes/{github_id}")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # 파일명으로 사용할 수 없는 문자 처리
    safe_name = repo_name.replace("/", "_")
    file_path = output_dir / f"{safe_name}.md"
    
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)
    return file_path

In [ ]:
# 사용법 예시
target_user = "input_your_github_id_here" # 여기에 GitHub ID를 입력하세요.

if target_user != "input_your_github_id_here":
    repos = get_user_repos(target_user)
    print(f"총 {len(repos)}개의 저장소를 찾았습니다.\n")

    for repo in repos:
        repo_full_name = repo['full_name']
        repo_name = repo['name']
        readme = get_readme_content(repo_full_name)
        if readme:
            file_path = save_readme(target_user, repo_name, readme)
            print(f"[{repo_full_name}] 저장 완료 -> {file_path}")
        else:
            print(f"[{repo_full_name}] README가 없거나 가져올 수 없습니다.")
else:
    print("target_user 변수에 GitHub ID를 입력해주세요.")